In [ ]:
!pip install pyngrok
!pip install fastapi

In [ ]:
!pip install fastapi python-multipart librosa transformers captum > /dev/null 2>&1

import os, io, gc
import torch
import librosa
import numpy as np
from fastapi import FastAPI, UploadFile, File
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2ForSequenceClassification
from captum.attr import IntegratedGradients

app = FastAPI()

# --- 1. MODEL INITIALIZATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Loading Wav2Vec2 on {device}...")

MODEL_PATH = "/content/drive/MyDrive/audio-20260509T124755Z-3-001/audio"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_PATH)
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_PATH).to(device).eval()

class Wav2Vec2Wrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_values):
        return self.model(input_values=input_values).logits

ig = IntegratedGradients(Wav2Vec2Wrapper(model))

# --- 2. MATH PROCESSORS ---
def extract_temporal_anomalies(attributions_np, window_size=1600):
    importance = np.abs(attributions_np)
    smoothed = np.convolve(importance, np.ones(window_size)/window_size, mode='same')
    if smoothed.max() > 0:
        smoothed = smoothed / smoothed.max()
    return smoothed

def generate_dynamic_explanation(probability, temporal_scores, sampling_rate=16000):
    if probability <= 0.5:
        return "Verified Authentic: The vocal tract frequencies flow naturally without robotic artifacts or unnatural splicing."

    peak_time_sec = int(np.argmax(temporal_scores)) / sampling_rate
    return f"AI Generation Detected: The model detected a breakdown in audio naturalness peaking around {peak_time_sec:.2f} seconds. High gradients indicate synthetic frequencies humans cannot produce naturally."

# --- 3. API ENDPOINT ---
@app.post("/analyze_audio")
async def analyze_audio(audio: UploadFile = File(...)):
    try:
        audio_bytes = await audio.read()
        audio_array, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)

        # Truncate to 10 seconds to save Colab VRAM
        if len(audio_array) > 16000 * 10:
            audio_array = audio_array[:16000 * 10]

        inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt", padding=True)
        input_values = inputs["input_values"].to(device)

        with torch.no_grad():
            logits = model(input_values).logits

        predicted_class_id = torch.argmax(logits, dim=-1).item()
        probs = torch.nn.functional.softmax(logits, dim=-1)

        is_fake = model.config.id2label[predicted_class_id].lower() == "fake"
        probability = probs[0][1].item() if is_fake else probs[0][1].item()

        input_values.requires_grad_()
        attributions = ig.attribute(inputs=input_values, target=predicted_class_id, n_steps=5)
        attr_np = attributions.squeeze().detach().cpu().numpy()

        torch.cuda.empty_cache()
        gc.collect()

        temporal_scores = extract_temporal_anomalies(attr_np)
        explanation = generate_dynamic_explanation(probability, temporal_scores)

        peak_idx = int(np.argmax(temporal_scores))
        peak_time_ms = int((peak_idx / 16000) * 1000)

        markers = []
        if probability > 0.5:
            markers.append({
                "timestampMs": peak_time_ms,
                "summary": "Synthetic Audio Peak Detected",
                "gridIndices": []
            })

        verdict_str = "AI Generated" if probability > 0.5 else "Authentic"
        conf_str = f"{(probability * 100 if probability > 0.5 else (1-probability)*100):.2f}%"

        return {
            "status": "success",
            "verdict": verdict_str,
            "confidence": conf_str,
            "details": explanation,
            "markers": markers
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

print("✅ Server Engine Ready! Now run Cell 2.")

🚀 Loading Wav2Vec2 on cuda...


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

✅ Server Engine Ready! Now run Cell 2.


In [ ]:
import cv2
import tempfile
import gc
from PIL import Image
from torchvision import transforms
from transformers import TimesformerModel, TimesformerConfig

# ==========================================
# 1. UPGRADED MODEL WITH XAI HOOKS
# ==========================================
class TemporalPhysicsEngine(torch.nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        config = TimesformerConfig(num_frames=16)
        self.backbone = TimesformerModel(config)

        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(768, 256),
            torch.nn.GELU(),
            torch.nn.Dropout(p=0.3),
            torch.nn.Linear(256, 1)
        )

    def forward(self, x, return_attention=False):
        # We explicitly request the attention maps from the Transformer
        outputs = self.backbone(pixel_values=x, output_attentions=return_attention)
        sequence_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(sequence_output)

        if return_attention:
            return logits, outputs.attentions
        return logits

# UPDATE THIS PATH!
TIMESFORMER_CKPT_PATH = "/content/drive/MyDrive/audio-20260509T124755Z-3-001/temporal_checkpoint3.pth"

# ==========================================
# 2. EXTRACTOR (NOW WITH TIMESTAMPS)
# ==========================================
def extract_16_frames_with_time(video_path):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0: return None, None

    step = max(total_frames // 16, 1)
    frames = []
    timestamps = []

    for i in range(16):
        frame_idx = min(i * step, total_frames - 1)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))
            # Grab the exact millisecond!
            timestamps.append(int(cap.get(cv2.CAP_PROP_POS_MSEC)))

    cap.release()

    while len(frames) < 16 and len(frames) > 0:
        frames.append(frames[-1])
        timestamps.append(timestamps[-1])
    return frames, timestamps

# ==========================================
# 3. THE XAI VIDEO ENDPOINT
# ==========================================
@app.post("/analyze_video")
async def analyze_video(video: UploadFile = File(...)):
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        with tempfile.NamedTemporaryFile(delete=False, suffix=".mp4") as temp_video:
            temp_video.write(await video.read())
            temp_video_path = temp_video.name

        pil_frames, timestamps = extract_16_frames_with_time(temp_video_path)
        os.remove(temp_video_path)

        if not pil_frames:
            return {"status": "error", "message": "Failed to extract 16 frames."}

        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        tensor_frames = [transform(frame) for frame in pil_frames]
        video_tensor = torch.stack(tensor_frames).unsqueeze(0).to(device)

        print("🎬 Loading TemporalPhysicsEngine to GPU...")
        video_model = TemporalPhysicsEngine(pretrained=False).to(device)
        checkpoint = torch.load(TIMESFORMER_CKPT_PATH, map_location=device, weights_only=True)
        video_model.load_state_dict(checkpoint['state_dict'])
        video_model.eval()

        # Inference with XAI turned ON
        with torch.no_grad(), torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            logits, attentions = video_model(video_tensor, return_attention=True)
            probability = torch.sigmoid(logits).item()

        verdict_str = "AI Generated" if probability > 0.5 else "Authentic"
        conf_str = f"{(probability * 100 if probability > 0.5 else (1-probability)*100):.2f}%"

        # ------------------------------------------
        # 🧪 FORENSIC MATH & MARKER GENERATION
        # ------------------------------------------
        markers = []
        details_text = "TimeSformer 3D Spatial-Temporal Analysis completed. No significant temporal jitter or spatial artifacts detected. The physical continuity is stable."

        if probability > 0.5 and len(attentions) > 0:
            # 1. Grab the attention from the very last layer
            last_attn = attentions[-1][0]
            # 2. Look at how the CLS token focuses on the video patches
            cls_attn = last_attn.mean(dim=0)[0, 1:]

            patches_per_frame = len(cls_attn) // 16
            temporal_scores = []

            # 3. Find the hottest frames
            for i in range(16):
                start = i * patches_per_frame
                end = start + patches_per_frame
                frame_score = cls_attn[start:end].max().item()
                max_patch_idx = cls_attn[start:end].argmax().item()
                temporal_scores.append((i, frame_score, max_patch_idx))

            # 4. Take the top 3 most manipulated frames
            temporal_scores.sort(key=lambda x: x[1], reverse=True)
            top_frames = temporal_scores[:3]

            anomaly_times = []
            for frame_idx, score, max_patch_idx in top_frames:
                ts_ms = timestamps[frame_idx]
                anomaly_times.append(f"{ts_ms/1000:.1f}s")

                grid_indices = []
                # 5. Map TimeSformer 14x14 vision patches to Android 16x16 Canvas!
                if patches_per_frame == 196:
                    r14 = max_patch_idx // 14
                    c14 = max_patch_idx % 14
                    r16 = int((r14 / 14.0) * 16)
                    c16 = int((c14 / 14.0) * 16)

                    # Create a 3x3 block around the anomaly so it's clearly visible on phone
                    for r in range(max(0, r16-1), min(16, r16+2)):
                        for c in range(max(0, c16-1), min(16, c16+2)):
                            grid_indices.append(r * 16 + c)

                markers.append({
                    "timestampMs": ts_ms,
                    "summary": "Structural Anomaly",
                    "gridIndices": grid_indices
                })

            # Update the dynamic text with exact timestamps
            details_text = f"TimeSformer deep analysis detected temporal inconsistencies. Peak anomalies isolated at {', '.join(anomaly_times)}, indicating localized structural distortions and AI generation artifacts."

        # Evict from GPU
        del video_model
        del video_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
        print("🧹 TemporalPhysicsEngine evicted from GPU.")

        return {
            "status": "success",
            "verdict": verdict_str,
            "confidence": conf_str,
            "details": details_text,
            "markers": markers
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
!fuser -k 8080/tcp

In [ ]:
import os
import signal
import socket
import threading
import time
import uvicorn
import nest_asyncio
import urllib.request

# 1. Configuration
PORT = 8888  # Switching to a fresh high port
nest_asyncio.apply()

# 2. Safety Check: Kill only the specific python process on this port if it exists
def kill_port(port):
    try:
        # Finding the process ID (PID) using the port
        pid = os.popen(f"lsof -t -i:{port}").read().strip()
        if pid:
            print(f"🧹 Clearing ghost process {pid} on port {port}...")
            os.kill(int(pid), signal.SIGTERM)
            time.sleep(2)
    except Exception as e:
        print(f"Note: Port {port} was already clear.")

kill_port(PORT)

# 3. Start the FastAPI Server
def run_app():
    try:
        config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
        server = uvicorn.Server(config)
        server.run()
    except Exception as e:
        print(f"❌ Server failed to start: {e}")

threading.Thread(target=run_app, daemon=True).start()
time.sleep(3) # Give it a moment to boot

# 4. Get Tunnel Credentials
try:
    external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
    print("\n" + "="*40)
    print(f"🔑 ENDPOINT IP: {external_ip}")
    print("="*40 + "\n")
except:
    print("Could not fetch IP. Use the one from your previous successful run.")

# 5. Open the Tunnel
print("⏳ Opening Tunnel... Use the link below:")
!npx localtunnel --port 8888

INFO:     Started server process [4515]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8888 (Press CTRL+C to quit)



🔑 ENDPOINT IP: 34.87.28.55

⏳ Opening Tunnel... Use the link below:
⠙⠹⠸⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://late-ears-boil.loca.lt
INFO:     106.222.177.242:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     106.222.177.242:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     106.222.177.242:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
🎬 Loading TemporalPhysicsEngine to GPU...
🧹 TemporalPhysicsEngine evicted from GPU.
INFO:     106.222.177.242:0 - "POST /analyze_video HTTP/1.1" 200 OK
🎬 Loading TemporalPhysicsEngine to GPU...
🧹 TemporalPhysicsEngine evicted from GPU.
INFO:     157.51.237.241:0 - "POST /analyze_video HTTP/1.1" 200 OK
^C
